# Encrypted Social Media Traffic Fingerprinting under Cross-Session Temporal Separation: A Public Benchmark and Multi-Flow Evaluation

This uses a **time-aligned concurrent multi-application replay as the primary validation/test protocol**. Each separately captured application session is shifted to a common relative start time (t=0), then all flows are merged by relative first-seen timestamp. This preserves each capture's internal flow timing, bursts, idle gaps, and ordering while simulating simultaneous application activity. It is simulated concurrency, not an originally captured simultaneous multi-app trace.

- **Days 1–3:** individual-flow training using all application classes.
- **Day 4:** time-aligned concurrent validation for common window selection and focal-model selection.
- **Day 5:** untouched time-aligned concurrent final test.
- Candidate windows: **W = [40,80,120,160,200,240,280,320,360,400]**.
- Buffers are populated by the model's **predicted class**, never by the ground-truth application label.
- The original homogeneous per-application block test is retained as a separate secondary analysis.

- **Multi-flow aggregation baseline:** majority voting is evaluated alongside probability averaging under the same time-aligned concurrent buffers, Day-4 validation windows, and locked Day-5 protocol. Because buffers are formed from predicted application identity, the majority-vote result is also reported as a deliberately simple hard-decision baseline rather than claimed as a novel method.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q lightgbm xgboost catboost shap lime

In [ ]:
import random, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, log_loss
)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/drive/MyDrive/mobile_merged_output')
RESULTS_DIR = Path('/content/drive/MyDrive/mobile_jcp_resultsMain3')
for name in ['metrics','per_class','confusion_matrices','plots','predictions','audits','mixed_app']:
    (RESULTS_DIR/name).mkdir(parents=True, exist_ok=True)

SEEDS = [42,123,456,789,2026]
WINDOW_SIZES = [40,80,120,160,200,240,280,320,360,400]
RARE_MIN_COUNT = 10
BOOTSTRAP_REPS = 500
BOOTSTRAP_SEED = 20260922


## Load data

In [ ]:
files = sorted([p for p in DATA_DIR.glob('*.csv') if 'summary' not in p.name.lower()])
if not files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR}')

frames = []
for p in files:
    d = pd.read_csv(p)
    if 'Label' not in d.columns:
        d['Label'] = p.stem.lower()
    if 'capture_file' not in d.columns:
        raise ValueError(f'{p.name} has no capture_file column')
    d['_original_row_order'] = np.arange(len(d))
    frames.append(d)
    print(f'{p.name}: {len(d):,} rows')

df = pd.concat(frames, ignore_index=True)
print('Combined:', df.shape)
display(df['Label'].value_counts().sort_index())

## Strict temporal split: Days 1–3 train, Day 4 validation, Day 5 test

The earlier within-capture 85/15 development split is removed. Each capture day now has a single role:

- **Days 1–3 → training**
- **Day 4 → validation/model and window selection**
- **Day 5 → final untouched test**

This prevents training and validation flows from coming from the same capture session. Multi-flow windows never cross capture boundaries.


In [ ]:
# STRICT COMPLETE-DAY TEMPORAL SPLIT
if 'capture_day' not in df.columns:
    raise ValueError("The dataset has no 'capture_day' column.")
if 'capture_file' not in df.columns:
    raise ValueError("The dataset has no 'capture_file' column.")

def normalize_day(value):
    s = str(value).strip().lower()
    digits = ''.join(ch for ch in s if ch.isdigit())
    return int(digits) if digits else np.nan

df['_day_number'] = df['capture_day'].map(normalize_day)
detected_days = set(df['_day_number'].dropna().astype(int).unique())
print('Detected capture days:', sorted(detected_days))
if detected_days != {1,2,3,4,5}:
    raise ValueError(f'Expected Days 1–5; found {sorted(detected_days)}')

# Establish chronological order inside every original capture.
order_candidates = [
    'bidirectional_first_seen_ms', 'src2dst_first_seen_ms',
    'dst2src_first_seen_ms', 'id', '_original_row_order'
]
ORDER_COL = next((c for c in order_candidates if c in df.columns), None)
if ORDER_COL is None:
    raise ValueError('No usable flow-order column was found.')

df['_flow_order_in_capture'] = -1
for capture, idx in df.groupby('capture_file', sort=False).groups.items():
    ordered_idx = df.loc[idx].sort_values([ORDER_COL, '_original_row_order']).index
    df.loc[ordered_idx, '_flow_order_in_capture'] = np.arange(len(ordered_idx))

# Complete-day roles.
df['_split'] = np.select(
    [df['_day_number'].isin([1,2,3]), df['_day_number'].eq(4), df['_day_number'].eq(5)],
    ['train','validation','test'], default='unused'
)
if df['_split'].eq('unused').any():
    raise RuntimeError('Some rows were not assigned to train/validation/test.')

split_audit = (
    df.groupby(['capture_file','Label','_day_number','_split'], as_index=False)
      .size().rename(columns={'size':'flows'})
      .sort_values(['_day_number','Label','capture_file'])
)
display(split_audit)
print('\nOverall split counts:')
display(df['_split'].value_counts())
print('\nClass counts by split:')
display(df.groupby(['Label','_split']).size().unstack(fill_value=0))

assert set(df.loc[df['_split'].eq('train'),'_day_number'].unique()) == {1,2,3}
assert set(df.loc[df['_split'].eq('validation'),'_day_number'].unique()) == {4}
assert set(df.loc[df['_split'].eq('test'),'_day_number'].unique()) == {5}

split_audit.to_csv(RESULTS_DIR/'audits'/'complete_day_split_audit.csv', index=False)
print('\nPASS: Days 1–3 train, Day 4 validation, Day 5 untouched test.')


In [ ]:
# AUDIT COMPLETE WINDOWS AVAILABLE ON DAY 4 AND DAY 5.
def complete_window_count_table(frame, split_name):
    rows=[]
    part=frame.loc[frame['_split'].eq(split_name)]
    for (capture,label), g in part.groupby(['capture_file','Label'], sort=False):
        n=len(g)
        for w in WINDOW_SIZES:
            complete=n//w
            retained=complete*w
            discarded=n-retained
            rows.append({
                'split':split_name,'capture_file':capture,'Class':label,'Window_size':w,
                'total_flows':n,'complete_windows':complete,'retained_flows':retained,
                'discarded_incomplete_flows':discarded,
                'retained_percent':100*retained/n if n else np.nan,
                'discarded_percent':100*discarded/n if n else np.nan
            })
    return pd.DataFrame(rows)

window_completeness = pd.concat([
    complete_window_count_table(df,'validation'),
    complete_window_count_table(df,'test')
], ignore_index=True)
window_completeness.to_csv(RESULTS_DIR/'audits'/'window_completeness_by_capture_class.csv', index=False)

display(window_completeness.groupby(['split','Window_size']).agg(
    complete_windows=('complete_windows','sum'),
    retained_flows=('retained_flows','sum'),
    discarded_incomplete_flows=('discarded_incomplete_flows','sum'),
    total_flows=('total_flows','sum')
).reset_index())


## Leakage-resistant feature preparation

In [ ]:
DROP_COLUMNS = {
    'Label','capture_day','capture_file','_day_number','_original_row_order','_split',
    '_flow_order_in_capture','id','expiration_id','application_name',
    'application_category_name','application_is_guessed','application_confidence',
    'requested_server_name','src_ip','dst_ip','src_mac','dst_mac','src_oui','dst_oui',
    'src_port','dst_port','splt_direction','splt_ps','splt_piat_ms'
}
ABSOLUTE_TIME_COLUMNS = {
    c for c in df.columns
    if c.endswith('_first_seen_ms') or c.endswith('_last_seen_ms')
}
DROP_COLUMNS |= ABSOLUTE_TIME_COLUMNS

X = df[[c for c in df.columns if c not in DROP_COLUMNS]].copy()
X = X.replace([np.inf,-np.inf], np.nan)

train_mask = df['_split'].eq('train')
train_view = X.loc[train_mask]
bad = list(train_view.columns[train_view.isna().all()])
bad += [c for c in train_view.columns if train_view[c].nunique(dropna=False) <= 1]
X = X.drop(columns=sorted(set(bad)))

le = LabelEncoder()
y = le.fit_transform(df['Label'])
class_names = le.classes_.tolist()
num_classes = len(class_names)

numeric_cols = X.select_dtypes(exclude=['object','string','category']).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print('Feature matrix:', X.shape)
print('Numeric:', len(numeric_cols))
print('Categorical:', len(categorical_cols))
print('Classes:', class_names)

## Preprocessing and models

In [ ]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=10, rare_token='__RARE__'):
        self.min_count = min_count
        self.rare_token = rare_token
    def fit(self, X, y=None):
        a = np.asarray(X, dtype=object)
        self.keep_ = []
        for j in range(a.shape[1]):
            s = pd.Series(a[:,j]).astype(str)
            vc = s.value_counts(dropna=False)
            self.keep_.append(set(vc[vc >= self.min_count].index))
        return self
    def transform(self, X):
        a = np.asarray(X, dtype=object).copy()
        for j, keep in enumerate(self.keep_):
            s = pd.Series(a[:,j]).astype(str)
            a[:,j] = np.where(s.isin(keep), s, self.rare_token)
        return a
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('rare', RareCategoryGrouper(RARE_MIN_COUNT)),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ], sparse_threshold=0)

def build_models(seed):
    return {
        'XGBoost': xgb.XGBClassifier(
            n_estimators=800, learning_rate=0.04, max_depth=6,
            min_child_weight=2, gamma=0.02, subsample=0.85,
            colsample_bytree=0.85, reg_alpha=0.05, reg_lambda=1.5,
            random_state=seed, n_jobs=-1, eval_metric='mlogloss',
            tree_method='hist', verbosity=0
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=800, learning_rate=0.04, num_leaves=31,
            max_depth=8, min_child_samples=20, subsample=0.85,
            subsample_freq=1, colsample_bytree=0.85, reg_alpha=0.05,
            reg_lambda=1.5, random_state=seed, n_jobs=-1,
            class_weight='balanced', verbose=-1
        ),
        'HistGradientBoosting': HistGradientBoostingClassifier(
            max_iter=700, learning_rate=0.05, max_leaf_nodes=31,
            max_depth=8, min_samples_leaf=20, l2_regularization=1.0,
            early_stopping=False, random_state=seed
        ),
        'CatBoost': CatBoostClassifier(
            iterations=800, learning_rate=0.05, depth=7,
            l2_leaf_reg=3.0, random_strength=0.5,
            random_seed=seed, verbose=0, allow_writing_files=False
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=600, max_depth=15, min_samples_split=8,
            min_samples_leaf=3, max_features='sqrt',
            random_state=seed, n_jobs=-1,
            class_weight='balanced_subsample'
        )
    }

## Multi-flow aggregation and operational measurements

Primary aggregation uses consecutive flows within each original application capture. The function also records the member indices of every complete window so sampling uncertainty, latency, packet/byte cost, and discarded-window behavior can be audited separately.


In [ ]:
def aggregate_windows(probabilities, true_labels, metadata, window_size):
    """Controlled homogeneous capture-block aggregation (secondary analysis only)."""
    probs_out, y_out, rows = [], [], []
    m = metadata.reset_index(drop=True).copy(); m['_p'] = np.arange(len(m))
    for capture, g in m.groupby('capture_file', sort=False):
        g = g.sort_values('_flow_order_in_capture'); pos = g['_p'].to_numpy()
        for start in range(0, len(pos), window_size):
            idx = pos[start:start+window_size]
            if len(idx) < window_size: continue
            labels = np.asarray(true_labels)[idx]
            if len(np.unique(labels)) != 1: raise RuntimeError('A capture contains multiple ground-truth labels.')
            probs_out.append(np.asarray(probabilities)[idx].mean(axis=0)); y_out.append(int(labels[0]))
            rows.append({'capture_file':capture,'window_size':window_size,
                         'window_start_order':int(g.iloc[start]['_flow_order_in_capture']),
                         'window_end_order':int(g.iloc[start+window_size-1]['_flow_order_in_capture']),
                         'member_positions':idx.tolist()})
    return np.asarray(probs_out), np.asarray(y_out,dtype=int), pd.DataFrame(rows)

def _time_column(metadata):
    """Return the best available first-seen timestamp column (NFStream timestamps are milliseconds)."""
    for c in ['bidirectional_first_seen_ms','src2dst_first_seen_ms','dst2src_first_seen_ms']:
        if c in metadata.columns:
            return c
    raise RuntimeError('Time-aligned concurrent replay requires a first-seen timestamp column.')

def time_aligned_concurrent_order(metadata):
    """Simulate simultaneous app activity by aligning each capture to t=0 and merging by relative flow time.

    Every original flow is retained. Within-capture timing, burstiness, idle gaps, and flow order are preserved.
    Only the absolute start time of each separately collected capture is shifted to a common origin.
    """
    m=metadata.reset_index(drop=True).copy(); m['_p']=np.arange(len(m))
    tcol=_time_column(m)
    m['_abs_t']=pd.to_numeric(m[tcol],errors='coerce')
    if m['_abs_t'].isna().any():
        raise RuntimeError(f'Missing/non-numeric timestamps found in {tcol}.')
    # Normalize EACH capture independently so all application sessions begin at a common t=0.
    m['_relative_time_ms']=m['_abs_t']-m.groupby('capture_file')['_abs_t'].transform('min')
    # Deterministic tie-breaking never uses the true class label.
    m=m.sort_values(['_relative_time_ms','capture_file','_flow_order_in_capture','_p'],kind='mergesort')
    return m['_p'].to_numpy(dtype=int), m.set_index('_p')['_relative_time_ms'].to_dict()

def predicted_buffer_concurrent_test(flow_probs,y_true,metadata,w):
    """Primary concurrent replay: time-align captures, merge by relative time, then route by predicted class only."""
    order,rel_time=time_aligned_concurrent_order(metadata)
    buffers=defaultdict(list); rows=[]; out_probs=[]; out_y=[]
    packet_col='bidirectional_packets' if 'bidirectional_packets' in metadata.columns else None
    byte_col='bidirectional_bytes' if 'bidirectional_bytes' in metadata.columns else None
    for stream_pos,idx in enumerate(order):
        route=int(np.argmax(flow_probs[idx]))
        buffers[route].append((idx,stream_pos,float(rel_time[idx])))
        if len(buffers[route])==w:
            members=buffers[route]; buffers[route]=[]
            ids=np.asarray([x[0] for x in members],dtype=int); labels=np.asarray(y_true)[ids]
            counts=np.bincount(labels,minlength=num_classes); majority=int(np.argmax(counts))
            purity=float(counts[majority]/len(labels)); avg=np.asarray(flow_probs)[ids].mean(axis=0)
            out_probs.append(avg); out_y.append(majority)
            r={'route_class':class_names[route],'majority_true_class':class_names[majority],
               'purity':purity,'mixed_true_classes':int(np.count_nonzero(counts)),
               'stream_start_position':members[0][1],'stream_end_position':members[-1][1],
               'relative_start_ms':members[0][2],'relative_end_ms':members[-1][2],
               'accumulation_latency_seconds':max(0.0,(members[-1][2]-members[0][2])/1000.0),
               'member_positions':ids.tolist()}
            if packet_col:
                r['packets_in_window']=pd.to_numeric(metadata.iloc[ids][packet_col],errors='coerce').sum()
            elif all(c in metadata.columns for c in ['src2dst_packets','dst2src_packets']):
                r['packets_in_window']=(pd.to_numeric(metadata.iloc[ids]['src2dst_packets'],errors='coerce').sum()+pd.to_numeric(metadata.iloc[ids]['dst2src_packets'],errors='coerce').sum())
            if byte_col:
                r['bytes_in_window']=pd.to_numeric(metadata.iloc[ids][byte_col],errors='coerce').sum()
            elif all(c in metadata.columns for c in ['src2dst_bytes','dst2src_bytes']):
                r['bytes_in_window']=(pd.to_numeric(metadata.iloc[ids]['src2dst_bytes'],errors='coerce').sum()+pd.to_numeric(metadata.iloc[ids]['dst2src_bytes'],errors='coerce').sum())
            rows.append(r)
    leftovers=sum(len(v) for v in buffers.values())
    return np.asarray(out_probs),np.asarray(out_y,dtype=int),pd.DataFrame(rows),leftovers

def metric_row(y_true, probs):
    """Metrics over evaluable classes only; absent classes are not inserted as zero-F1 classes."""
    y_true=np.asarray(y_true,dtype=int); pred=np.asarray(probs).argmax(axis=1)
    present=np.unique(y_true)
    return {
        'Accuracy':accuracy_score(y_true,pred),
        'Precision_macro':precision_score(y_true,pred,labels=present,average='macro',zero_division=0),
        'Recall_macro':recall_score(y_true,pred,labels=present,average='macro',zero_division=0),
        'F1_macro':f1_score(y_true,pred,labels=present,average='macro',zero_division=0),
        'F1_weighted':f1_score(y_true,pred,labels=present,average='weighted',zero_division=0),
        'LogLoss':log_loss(y_true,probs,labels=np.arange(num_classes)),
        'Windows':len(y_true), 'Evaluable_classes':len(present)
    }

def per_class_rows(y_true, probs, seed, model_name, w, protocol):
    pred=np.asarray(probs).argmax(axis=1); rows=[]
    for cid,cls in enumerate(class_names):
        support=int(np.sum(np.asarray(y_true)==cid))
        if support==0:
            rows.append({'Seed':seed,'Model':model_name,'Window_size':w,'Class':cls,
                         'Precision':np.nan,'Recall':np.nan,'F1':np.nan,'Support':0,'Evaluable':False,'Protocol':protocol})
        else:
            rows.append({'Seed':seed,'Model':model_name,'Window_size':w,'Class':cls,
                         'Precision':precision_score(y_true,pred,labels=[cid],average='macro',zero_division=0),
                         'Recall':recall_score(y_true,pred,labels=[cid],average='macro',zero_division=0),
                         'F1':f1_score(y_true,pred,labels=[cid],average='macro',zero_division=0),
                         'Support':support,'Evaluable':True,'Protocol':protocol})
    return rows

def stratified_bootstrap_indices(y_true, rng):
    y_true=np.asarray(y_true); pieces=[]
    for cls in np.unique(y_true):
        idx=np.where(y_true==cls)[0]; pieces.append(rng.choice(idx,size=len(idx),replace=True))
    return np.concatenate(pieces)


## Day-4 time-aligned concurrent-application window selection with sampling uncertainty

Candidate windows are evaluated on a time-aligned concurrent Day-4 stream. Flows are routed to buffers by the model's **predicted class**, not by ground truth. The common window is selected using mean Macro-F1 across models/seeds. Macro metrics include only classes with positive support at that operating point. Bootstrap intervals quantify sampling uncertainty. The highest-performing model at the locked window is automatically selected for SHAP/LIME.


In [ ]:
train_idx=np.where(df['_split'].eq('train'))[0]; val_idx=np.where(df['_split'].eq('validation'))[0]; test_idx=np.where(df['_split'].eq('test'))[0]
Xtr_raw,Xv_raw,Xte_raw=X.iloc[train_idx],X.iloc[val_idx],X.iloc[test_idx]
ytr,yv,yte=y[train_idx],y[val_idx],y[test_idx]
meta_cols=['capture_file','Label','_flow_order_in_capture']
for c in ['bidirectional_first_seen_ms','bidirectional_last_seen_ms','src2dst_first_seen_ms','dst2src_first_seen_ms','bidirectional_packets','bidirectional_bytes','src2dst_packets','dst2src_packets','src2dst_bytes','dst2src_bytes']:
    if c in df.columns and c not in meta_cols: meta_cols.append(c)
meta_val=df.iloc[val_idx][meta_cols].reset_index(drop=True); meta_test=df.iloc[test_idx][meta_cols].reset_index(drop=True)

def fit_model(model_name,model,Xtr,ytr,Xv,yv):
    if model_name=='XGBoost': model.fit(Xtr,ytr,eval_set=[(Xv,yv)],verbose=False)
    elif model_name=='LightGBM': model.fit(Xtr,ytr,eval_set=[(Xv,yv)],callbacks=[lgb.early_stopping(40,verbose=False)])
    elif model_name=='CatBoost': model.fit(Xtr,ytr,eval_set=(Xv,yv),early_stopping_rounds=40,verbose=False)
    else: model.fit(Xtr,ytr)
    return model

validation_metric_rows=[]; validation_class_rows=[]; validation_window_cache={}; validation_flow_probability_cache={}; test_probability_cache={}
for seed in SEEDS:
    print(); print('='*80); print('SEED',seed); print('='*80); np.random.seed(seed); random.seed(seed)
    prep=make_preprocessor(); Xtr=prep.fit_transform(Xtr_raw); Xv=prep.transform(Xv_raw); Xte=prep.transform(Xte_raw)
    for model_name,model in build_models(seed).items():
        print('Training',model_name); model=fit_model(model_name,model,Xtr,ytr,Xv,yv); val_probs=model.predict_proba(Xv)
        validation_flow_probability_cache[(seed,model_name)]=val_probs
        for w in WINDOW_SIZES:
            vp,vy,vm,leftovers=predicted_buffer_concurrent_test(val_probs,yv,meta_val,w)
            if len(vy)==0: continue
            validation_window_cache[(seed,model_name,w)]={'probs':vp,'y':vy,'meta':vm}
            row={'Seed':seed,'Model':model_name,'Window_size':w,**metric_row(vy,vp),
                 'Mean_buffer_purity':vm['purity'].mean(),'Incomplete_buffered_flows':leftovers}
            validation_metric_rows.append(row)
            validation_class_rows.extend(per_class_rows(vy,vp,seed,model_name,w,'time_aligned_concurrent_validation'))
            print(f'  Concurrent Day4 W={w}: macroF1={row["F1_macro"]:.4f}, windows={row["Windows"]}, classes={row["Evaluable_classes"]}')
        test_probability_cache[(seed,model_name)]=model.predict_proba(Xte)

validation_metrics_df=pd.DataFrame(validation_metric_rows); validation_per_class_df=pd.DataFrame(validation_class_rows)
validation_metrics_df.to_csv(RESULTS_DIR/'metrics'/'day4_mixed_all_seed_window_metrics.csv',index=False)
validation_per_class_df.to_csv(RESULTS_DIR/'per_class'/'day4_mixed_window_per_class_metrics.csv',index=False)
validation_model_window=(validation_metrics_df.groupby(['Model','Window_size'],as_index=False).agg(
    Validation_F1_macro_mean=('F1_macro','mean'),Validation_F1_macro_std=('F1_macro','std'),
    Validation_accuracy_mean=('Accuracy','mean'),Validation_windows_mean=('Windows','mean'),
    Validation_evaluable_classes_mean=('Evaluable_classes','mean'),Mean_buffer_purity=('Mean_buffer_purity','mean')))
common_window_scores=(validation_model_window.groupby('Window_size',as_index=False).agg(
    Common_validation_macro_F1=('Validation_F1_macro_mean','mean'),Common_validation_accuracy=('Validation_accuracy_mean','mean'),
    Models_present=('Model','nunique'),Evaluable_classes_mean=('Validation_evaluable_classes_mean','mean'),Mean_buffer_purity=('Mean_buffer_purity','mean')))
common_window_scores=common_window_scores[common_window_scores.Models_present.eq(5)].sort_values(['Common_validation_macro_F1','Window_size'],ascending=[False,True]).reset_index(drop=True)
if common_window_scores.empty: raise RuntimeError('No candidate mixed window has all five models.')
LOCKED_WINDOW=int(common_window_scores.iloc[0].Window_size)
validation_locked_model=(validation_model_window[validation_model_window.Window_size.eq(LOCKED_WINDOW)]
    .sort_values(['Validation_F1_macro_mean','Validation_accuracy_mean','Model'],ascending=[False,False,True]).reset_index(drop=True))
LOCKED_MODEL=str(validation_locked_model.iloc[0].Model); TEST_SELECTED_MODEL=LOCKED_MODEL

rng=np.random.default_rng(BOOTSTRAP_SEED); boot_scores={w:[] for w in WINDOW_SIZES}; selection_counts={w:0 for w in WINDOW_SIZES}
for b in range(BOOTSTRAP_REPS):
    rep_scores={}
    for w in WINDOW_SIZES:
        vals=[]
        for seed in SEEDS:
            for model_name in build_models(seed).keys():
                item=validation_window_cache.get((seed,model_name,w))
                if item is None: continue
                idx=stratified_bootstrap_indices(item['y'],rng); pred=item['probs'][idx].argmax(axis=1); present=np.unique(item['y'][idx])
                vals.append(f1_score(item['y'][idx],pred,labels=present,average='macro',zero_division=0))
        if vals: rep_scores[w]=float(np.mean(vals)); boot_scores[w].append(rep_scores[w])
    if rep_scores: selection_counts[max(rep_scores,key=lambda x:(rep_scores[x],-x))]+=1
uncertainty_rows=[]
for w in WINDOW_SIZES:
    vals=np.asarray(boot_scores[w])
    if len(vals): uncertainty_rows.append({'Window_size':w,'Bootstrap_mean_macro_F1':vals.mean(),'CI95_low':np.quantile(vals,.025),'CI95_high':np.quantile(vals,.975),'Selection_frequency':selection_counts[w]/BOOTSTRAP_REPS})
common_window_scores=common_window_scores.merge(pd.DataFrame(uncertainty_rows),on='Window_size',how='left')
per_class_sensitivity=(validation_per_class_df.groupby(['Class','Window_size'],as_index=False).agg(F1_mean=('F1','mean'),F1_std=('F1','std'),Support_mean=('Support','mean')))
per_class_sensitivity.to_csv(RESULTS_DIR/'per_class'/'day4_mixed_per_class_window_sensitivity.csv',index=False)
validation_model_window.to_csv(RESULTS_DIR/'metrics'/'day4_mixed_model_window_summary.csv',index=False)
common_window_scores.to_csv(RESULTS_DIR/'metrics'/'day4_mixed_common_window_selection_with_bootstrap.csv',index=False)
validation_locked_model.to_csv(RESULTS_DIR/'metrics'/'day4_mixed_locked_model_for_xai.csv',index=False)
with open(RESULTS_DIR/'metrics'/'locked_window.txt','w') as f: f.write(str(LOCKED_WINDOW))
with open(RESULTS_DIR/'metrics'/'locked_xai_model.txt','w') as f: f.write(LOCKED_MODEL)
print(); print('DAY-4 MIXED COMMON WINDOW SELECTION'); display(common_window_scores)
print(f'LOCKED WINDOW: W={LOCKED_WINDOW}'); print(f'LOCKED XAI FOCAL MODEL: {LOCKED_MODEL}')

# PRIMARY FINAL DAY-5 TEST: mixed application stream, W=1 and validation-locked W* only.
FINAL_TEST_WINDOWS=sorted(set([1,LOCKED_WINDOW])); metric_rows=[]; class_rows=[]; cms={}; mixed_window_rows=[]
for seed in SEEDS:
    for model_name in build_models(seed).keys():
        flow_probs=test_probability_cache[(seed,model_name)]
        for w in FINAL_TEST_WINDOWS:
            wp,wy,wm,leftovers=predicted_buffer_concurrent_test(flow_probs,yte,meta_test,w)
            if len(wy)==0: raise RuntimeError(f'No mixed Day-5 windows for W={w}.')
            metrics=metric_row(wy,wp); metric_rows.append({'Seed':seed,'Model':model_name,'Window_size':w,**metrics,
                'Mean_buffer_purity':wm['purity'].mean(),'Median_buffer_purity':wm['purity'].median(),'Incomplete_buffered_flows':leftovers})
            pred=wp.argmax(axis=1); class_rows.extend(per_class_rows(wy,wp,seed,model_name,w,'time_aligned_concurrent_test'))
            cms.setdefault((model_name,w),[]).append(confusion_matrix(wy,pred,labels=np.arange(num_classes)))
            mm=wm.drop(columns=['member_positions']).copy(); mm['Seed']=seed; mm['Model']=model_name; mm['Window_size']=w; mixed_window_rows.append(mm)
            print(f'Concurrent Day5 {model_name} seed={seed} W={w}: macroF1={metrics["F1_macro"]:.4f}, windows={metrics["Windows"]}, classes={metrics["Evaluable_classes"]}')
metrics_df=pd.DataFrame(metric_rows); per_class_df=pd.DataFrame(class_rows)
mixed_windows=pd.concat(mixed_window_rows,ignore_index=True) if mixed_window_rows else pd.DataFrame()
metrics_df.to_csv(RESULTS_DIR/'mixed_app'/'day5_mixed_primary_all_seed_metrics.csv',index=False)
per_class_df.to_csv(RESULTS_DIR/'mixed_app'/'day5_mixed_primary_per_class_metrics.csv',index=False)
mixed_windows.to_csv(RESULTS_DIR/'mixed_app'/'day5_mixed_primary_buffer_details.csv',index=False)
print(); print('PASS: time-aligned concurrent Day 5 was scored only after time-aligned concurrent Day-4 window/model locks were established.')


In [ ]:
# ---------------------------------------------------------------
# PRIMARY CONCURRENT INCOMPLETE-BUFFER / COVERAGE SUMMARY
# ---------------------------------------------------------------

coverage_rows = []

# Day 4: locked window only
day4_cov = validation_metrics_df[
    validation_metrics_df['Window_size'].eq(LOCKED_WINDOW)
].copy()

coverage_rows.append({
    'Split': 'Day 4',
    'Window_size': LOCKED_WINDOW,
    'Total_flows': len(yv),
    'Completed_windows_mean': day4_cov['Windows'].mean(),
    'Completed_windows_std': day4_cov['Windows'].std(),
    'Incomplete_flows_mean': day4_cov['Incomplete_buffered_flows'].mean(),
    'Incomplete_flows_std': day4_cov['Incomplete_buffered_flows'].std(),
    'Incomplete_percent_mean':
        100 * day4_cov['Incomplete_buffered_flows'].mean() / len(yv),
    'Incomplete_percent_std':
        100 * day4_cov['Incomplete_buffered_flows'].std() / len(yv)
})

# Day 5: W=1 and locked W*
for w in [1, LOCKED_WINDOW]:
    day5_cov = metrics_df[
        metrics_df['Window_size'].eq(w)
    ].copy()

    coverage_rows.append({
        'Split': 'Day 5',
        'Window_size': w,
        'Total_flows': len(yte),
        'Completed_windows_mean': day5_cov['Windows'].mean(),
        'Completed_windows_std': day5_cov['Windows'].std(),
        'Incomplete_flows_mean':
            day5_cov['Incomplete_buffered_flows'].mean(),
        'Incomplete_flows_std':
            day5_cov['Incomplete_buffered_flows'].std(),
        'Incomplete_percent_mean':
            100 * day5_cov['Incomplete_buffered_flows'].mean() / len(yte),
        'Incomplete_percent_std':
            100 * day5_cov['Incomplete_buffered_flows'].std() / len(yte)
    })

concurrent_coverage_summary = pd.DataFrame(coverage_rows)

concurrent_coverage_summary.to_csv(
    RESULTS_DIR /
    'mixed_app' /
    'concurrent_incomplete_buffer_summary.csv',
    index=False
)

display(concurrent_coverage_summary)

## Sliding-window sensitivity analysis for incomplete-window robustness



In [ ]:
# ------------------------------------------------------------------
# REVIEWER-3 SENSITIVITY: 50% overlapping windows at LOCKED W* only
# ------------------------------------------------------------------

def predicted_buffer_concurrent_sliding_test(flow_probs, y_true, metadata, w, overlap_fraction=0.50):
    """Time-aligned concurrent replay with predicted-class routing and overlapping windows.

    This is a post-lock sensitivity analysis. Flows are first ordered exactly as in the
    primary concurrent replay and routed using ONLY the model's predicted class. Within
    each predicted-class route buffer, windows of size w are emitted with 50% overlap.
    Ground-truth labels are used only after a window is formed to define its evaluation
    target (majority true class), exactly as in the primary mixed-buffer evaluation.
    """
    if w <= 1:
        stride = 1
    else:
        stride = max(1, int(round(w * (1.0 - overlap_fraction))))

    order, rel_time = time_aligned_concurrent_order(metadata)
    routed = defaultdict(list)

    # Build each deployable route stream using predicted class only.
    for stream_pos, idx in enumerate(order):
        route = int(np.argmax(flow_probs[idx]))
        routed[route].append((idx, stream_pos, float(rel_time[idx])))

    packet_col = 'bidirectional_packets' if 'bidirectional_packets' in metadata.columns else None
    byte_col = 'bidirectional_bytes' if 'bidirectional_bytes' in metadata.columns else None
    out_probs, out_y, rows = [], [], []
    covered_positions = set()

    for route, members in routed.items():
        if len(members) < w:
            continue
        for start in range(0, len(members) - w + 1, stride):
            win = members[start:start+w]
            ids = np.asarray([x[0] for x in win], dtype=int)
            labels = np.asarray(y_true)[ids]
            counts = np.bincount(labels, minlength=num_classes)
            majority = int(np.argmax(counts))
            purity = float(counts[majority] / len(labels))
            avg = np.asarray(flow_probs)[ids].mean(axis=0)

            out_probs.append(avg)
            out_y.append(majority)
            covered_positions.update(ids.tolist())

            r = {
                'route_class': class_names[route],
                'majority_true_class': class_names[majority],
                'purity': purity,
                'mixed_true_classes': int(np.count_nonzero(counts)),
                'stream_start_position': win[0][1],
                'stream_end_position': win[-1][1],
                'relative_start_ms': win[0][2],
                'relative_end_ms': win[-1][2],
                'accumulation_latency_seconds': max(0.0, (win[-1][2] - win[0][2]) / 1000.0),
                'member_positions': ids.tolist(),
                'stride': stride,
                'overlap_fraction': overlap_fraction,
            }
            if packet_col:
                r['packets_in_window'] = pd.to_numeric(metadata.iloc[ids][packet_col], errors='coerce').sum()
            elif all(c in metadata.columns for c in ['src2dst_packets','dst2src_packets']):
                r['packets_in_window'] = (
                    pd.to_numeric(metadata.iloc[ids]['src2dst_packets'], errors='coerce').sum() +
                    pd.to_numeric(metadata.iloc[ids]['dst2src_packets'], errors='coerce').sum()
                )
            if byte_col:
                r['bytes_in_window'] = pd.to_numeric(metadata.iloc[ids][byte_col], errors='coerce').sum()
            elif all(c in metadata.columns for c in ['src2dst_bytes','dst2src_bytes']):
                r['bytes_in_window'] = (
                    pd.to_numeric(metadata.iloc[ids]['src2dst_bytes'], errors='coerce').sum() +
                    pd.to_numeric(metadata.iloc[ids]['dst2src_bytes'], errors='coerce').sum()
                )
            rows.append(r)

    total_flows = len(metadata)
    covered = len(covered_positions)
    uncovered = total_flows - covered
    coverage_pct = 100.0 * covered / total_flows if total_flows else np.nan
    return (np.asarray(out_probs), np.asarray(out_y, dtype=int), pd.DataFrame(rows),
            uncovered, covered, coverage_pct, stride)

sliding_rows = []
sliding_class_rows = []
sliding_details = []

# IMPORTANT: LOCKED_WINDOW has already been selected using Day-4 PRIMARY non-overlapping validation.
# This block evaluates Day 5 only at that already-locked operating point.
for seed in SEEDS:
    for model_name in build_models(seed).keys():
        flow_probs = test_probability_cache[(seed, model_name)]
        sp, sy, sm, uncovered, covered, coverage_pct, stride = predicted_buffer_concurrent_sliding_test(
            flow_probs, yte, meta_test, LOCKED_WINDOW, overlap_fraction=0.50
        )
        if len(sy) == 0:
            raise RuntimeError(f'No sliding Day-5 windows for {model_name}, seed={seed}, W={LOCKED_WINDOW}.')

        mr = metric_row(sy, sp)
        sliding_rows.append({
            'Seed': seed,
            'Model': model_name,
            'Window_size': LOCKED_WINDOW,
            'Stride': stride,
            'Overlap_fraction': 0.50,
            **mr,
            'Mean_buffer_purity': sm['purity'].mean(),
            'Median_buffer_purity': sm['purity'].median(),
            'Unique_flows_covered': covered,
            'Unique_flows_uncovered': uncovered,
            'Unique_flow_coverage_pct': coverage_pct,
        })
        sliding_class_rows.extend(
            per_class_rows(sy, sp, seed, model_name, LOCKED_WINDOW, 'time_aligned_concurrent_sliding_50pct_test')
        )
        sd = sm.drop(columns=['member_positions']).copy()
        sd['Seed'] = seed
        sd['Model'] = model_name
        sd['Window_size'] = LOCKED_WINDOW
        sliding_details.append(sd)

sliding_metrics_df = pd.DataFrame(sliding_rows)
sliding_per_class_df = pd.DataFrame(sliding_class_rows)
sliding_details_df = pd.concat(sliding_details, ignore_index=True) if sliding_details else pd.DataFrame()

sliding_metrics_df.to_csv(
    RESULTS_DIR/'mixed_app'/'day5_sliding_50pct_locked_sensitivity_all_seeds.csv', index=False
)
sliding_per_class_df.to_csv(
    RESULTS_DIR/'per_class'/'day5_sliding_50pct_locked_sensitivity_per_class.csv', index=False
)
sliding_details_df.to_csv(
    RESULTS_DIR/'audits'/'day5_sliding_50pct_locked_window_details.csv', index=False
)

sliding_summary = sliding_metrics_df.groupby(['Model','Window_size','Stride'], as_index=False).agg(
    Accuracy_mean=('Accuracy','mean'), Accuracy_std=('Accuracy','std'),
    Precision_macro_mean=('Precision_macro','mean'), Precision_macro_std=('Precision_macro','std'),
    Recall_macro_mean=('Recall_macro','mean'), Recall_macro_std=('Recall_macro','std'),
    F1_macro_mean=('F1_macro','mean'), F1_macro_std=('F1_macro','std'),
    F1_weighted_mean=('F1_weighted','mean'), F1_weighted_std=('F1_weighted','std'),
    LogLoss_mean=('LogLoss','mean'), LogLoss_std=('LogLoss','std'),
    Windows_mean=('Windows','mean'), Evaluable_classes_mean=('Evaluable_classes','mean'),
    Unique_flows_covered_mean=('Unique_flows_covered','mean'),
    Unique_flows_uncovered_mean=('Unique_flows_uncovered','mean'),
    Unique_flow_coverage_pct_mean=('Unique_flow_coverage_pct','mean'),
    Mean_buffer_purity=('Mean_buffer_purity','mean')
)

# Compare directly with the PRIMARY non-overlapping result at the SAME locked W*.
primary_locked = metrics_df[metrics_df['Window_size'].eq(LOCKED_WINDOW)].groupby('Model', as_index=False).agg(
    Primary_nonoverlap_F1_macro=('F1_macro','mean'),
    Primary_nonoverlap_windows=('Windows','mean'),
    Primary_incomplete_buffered_flows=('Incomplete_buffered_flows','mean')
)
sliding_compare = sliding_summary.merge(primary_locked, on='Model', how='left')
sliding_compare['Macro_F1_difference_sliding_minus_primary'] = (
    sliding_compare['F1_macro_mean'] - sliding_compare['Primary_nonoverlap_F1_macro']
)

sliding_summary.to_csv(
    RESULTS_DIR/'mixed_app'/'day5_sliding_50pct_locked_sensitivity_summary.csv', index=False
)
sliding_compare.to_csv(
    RESULTS_DIR/'mixed_app'/'day5_sliding_50pct_vs_primary_comparison.csv', index=False
)

print('POST-LOCK SENSITIVITY ONLY — this analysis did not select W*.')
print(f'Locked W* = {LOCKED_WINDOW}; 50% overlap stride = {max(1, int(round(LOCKED_WINDOW * 0.5)))}')
display(sliding_compare)


## Incomplete-window retention and operational latency/cost

For each Day-4 and Day-5 capture, this section reports how many flows form complete windows and how many are discarded as incomplete remainders. It also measures the time required to accumulate a complete window. When packet/byte totals are present in the dataset, their distributions are reported as well.


In [ ]:
# Operational cost of the PRIMARY time-aligned concurrent protocol.
# Uses the actual predicted-route buffers created during validation/test, so latency reflects
# how long a deployment would wait for W flows assigned to the same predicted application buffer.
operational_parts=[]
for (seed,model_name,w),item in validation_window_cache.items():
    z=item['meta'].copy()
    if len(z):
        z['split']='validation'; z['Seed']=seed; z['Model']=model_name; z['Window_size']=w
        operational_parts.append(z)
if len(mixed_windows):
    z=mixed_windows.copy(); z['split']='test'; operational_parts.append(z)
operational_windows=pd.concat(operational_parts,ignore_index=True) if operational_parts else pd.DataFrame()
operational_windows.to_csv(RESULTS_DIR/'audits'/'concurrent_window_operational_measurements.csv',index=False)

summary_aggs={}
for col in ['accumulation_latency_seconds','packets_in_window','bytes_in_window']:
    if col in operational_windows.columns:
        summary_aggs[col+'_median']=(col,'median')
        summary_aggs[col+'_q1']=(col,lambda s:s.quantile(.25))
        summary_aggs[col+'_q3']=(col,lambda s:s.quantile(.75))
operational_summary=(operational_windows.groupby(['split','Window_size'],as_index=False).agg(**summary_aggs) if summary_aggs else pd.DataFrame())
operational_summary.to_csv(RESULTS_DIR/'audits'/'concurrent_window_operational_summary_median_iqr.csv',index=False)
display(operational_summary)


## Secondary controlled homogeneous per-application test

This block retains the original same-application capture-window experiment as a **separate secondary result**. It is not used for the main Day-5 conclusions, confusion matrices, per-class tables, plots, or XAI model selection. Classes unable to form a complete locked window are reported as N/A and excluded from Macro-F1 rather than being assigned artificial zero scores.


In [ ]:
homogeneous_metric_rows=[]; homogeneous_class_rows=[]
for seed in SEEDS:
    for model_name in build_models(seed).keys():
        probs=test_probability_cache[(seed,model_name)]
        for w in FINAL_TEST_WINDOWS:
            hp,hy,hm=aggregate_windows(probs,yte,meta_test,w)
            if len(hy)==0: continue
            mr=metric_row(hy,hp); homogeneous_metric_rows.append({'Seed':seed,'Model':model_name,'Window_size':w,**mr})
            homogeneous_class_rows.extend(per_class_rows(hy,hp,seed,model_name,w,'homogeneous_secondary'))
homogeneous_metrics_df=pd.DataFrame(homogeneous_metric_rows); homogeneous_per_class_df=pd.DataFrame(homogeneous_class_rows)
homogeneous_metrics_df.to_csv(RESULTS_DIR/'metrics'/'day5_homogeneous_secondary_all_seed_metrics.csv',index=False)
homogeneous_per_class_df.to_csv(RESULTS_DIR/'per_class'/'day5_homogeneous_secondary_per_class_metrics.csv',index=False)
homogeneous_summary=homogeneous_metrics_df.groupby(['Model','Window_size']).agg(
    Accuracy_mean=('Accuracy','mean'),Accuracy_std=('Accuracy','std'),Precision_macro_mean=('Precision_macro','mean'),Precision_macro_std=('Precision_macro','std'),
    Recall_macro_mean=('Recall_macro','mean'),Recall_macro_std=('Recall_macro','std'),F1_macro_mean=('F1_macro','mean'),F1_macro_std=('F1_macro','std'),
    F1_weighted_mean=('F1_weighted','mean'),F1_weighted_std=('F1_weighted','std'),LogLoss_mean=('LogLoss','mean'),LogLoss_std=('LogLoss','std'),
    Windows_mean=('Windows','mean'),Evaluable_classes_mean=('Evaluable_classes','mean')).reset_index()
display(homogeneous_summary)


## Primary time-aligned concurrent Day-5 mean ± standard deviation

The table below is the **main final test result**. It reports the single-flow baseline and the Day-4-selected locked mixed-buffer operating point with full metrics across five seeds. Macro metrics exclude classes with zero support in the evaluated windows; the number of evaluable classes is reported explicitly.


In [ ]:
agg = metrics_df.groupby(['Model','Window_size']).agg(
    Accuracy_mean=('Accuracy','mean'), Accuracy_std=('Accuracy','std'),
    Precision_macro_mean=('Precision_macro','mean'), Precision_macro_std=('Precision_macro','std'),
    Recall_macro_mean=('Recall_macro','mean'), Recall_macro_std=('Recall_macro','std'),
    F1_macro_mean=('F1_macro','mean'), F1_macro_std=('F1_macro','std'),
    F1_weighted_mean=('F1_weighted','mean'), F1_weighted_std=('F1_weighted','std'),
    LogLoss_mean=('LogLoss','mean'), LogLoss_std=('LogLoss','std'),
    Windows_mean=('Windows','mean'), Evaluable_classes_mean=('Evaluable_classes','mean'),
    Mean_buffer_purity=('Mean_buffer_purity','mean'), Incomplete_buffered_flows_mean=('Incomplete_buffered_flows','mean')
).reset_index()
for metric in ['Accuracy','Precision_macro','Recall_macro','F1_macro','F1_weighted','LogLoss']:
    agg[f'{metric}_mean_std']=agg[f'{metric}_mean'].map(lambda x:f'{x:.4f}')+' ± '+agg[f'{metric}_std'].fillna(0).map(lambda x:f'{x:.4f}')
display_cols=['Model','Window_size','Accuracy_mean_std','Precision_macro_mean_std','Recall_macro_mean_std','F1_macro_mean_std','F1_weighted_mean_std','LogLoss_mean_std','Windows_mean','Evaluable_classes_mean','Mean_buffer_purity','Incomplete_buffered_flows_mean']
summary=agg[display_cols].sort_values(['Window_size','Model'])
summary.to_csv(RESULTS_DIR/'mixed_app'/'day5_mixed_primary_summary_mean_std.csv',index=False); display(summary)


## Multi-flow aggregation baseline — majority voting

**Important interpretation:** concurrent buffers are routed using each flow’s predicted class. Consequently, member hard predictions within a completed route buffer are normally identical. Majority voting is therefore a strict hard-decision baseline for the current deployable grouping rule; it tests whether retaining the soft probability information adds value beyond the routing decisions themselves.


In [ ]:
# Majority-vote multi-flow baseline using the EXACT SAME completed concurrent buffers.

def majority_vote_probs_from_buffer_details(flow_probs, buffer_meta):
    out=[]
    for ids in buffer_meta['member_positions']:
        ids=np.asarray(ids,dtype=int)
        hard=np.asarray(flow_probs)[ids].argmax(axis=1)
        counts=np.bincount(hard,minlength=num_classes).astype(float)
        # Vote proportions retain a valid probability vector for LogLoss.
        out.append(counts/counts.sum())
    return np.asarray(out)

baseline_val_rows=[]
baseline_test_rows=[]

# Day-4: compare aggregation rules across every candidate window.
for (seed,model_name,w),item in validation_window_cache.items():
    flow_probs = None
    # Recover the fitted-model Day-4 flow probabilities from the member windows themselves is not possible,
    # so use the cached completed-window member positions plus a dedicated cache created below if available.
    if 'validation_flow_probability_cache' not in globals():
        raise RuntimeError('validation_flow_probability_cache is required. Re-run the training cell after applying this notebook update.')
    flow_probs=validation_flow_probability_cache[(seed,model_name)]
    mv_probs=majority_vote_probs_from_buffer_details(flow_probs,item['meta'])
    mr=metric_row(item['y'],mv_probs)
    baseline_val_rows.append({'Seed':seed,'Model':model_name,'Window_size':w,'Aggregation_Method':'Majority_Vote',**mr})
    pa=metric_row(item['y'],item['probs'])
    baseline_val_rows.append({'Seed':seed,'Model':model_name,'Window_size':w,'Aggregation_Method':'Probability_Average',**pa})

# Day-5: compare only W=1 and locked W*, exactly as in the primary test.
for seed in SEEDS:
    for model_name in build_models(seed).keys():
        flow_probs=test_probability_cache[(seed,model_name)]
        for w in FINAL_TEST_WINDOWS:
            pa_probs,wy,wm,leftovers=predicted_buffer_concurrent_test(flow_probs,yte,meta_test,w)
            mv_probs=majority_vote_probs_from_buffer_details(flow_probs,wm)
            for method,probs in [('Probability_Average',pa_probs),('Majority_Vote',mv_probs)]:
                mr=metric_row(wy,probs)
                baseline_test_rows.append({'Seed':seed,'Model':model_name,'Window_size':w,'Aggregation_Method':method,**mr})

baseline_validation_df=pd.DataFrame(baseline_val_rows)
baseline_day5_df=pd.DataFrame(baseline_test_rows)
baseline_validation_df.to_csv(RESULTS_DIR/'metrics'/'day4_aggregation_baseline_comparison.csv',index=False)
baseline_day5_df.to_csv(RESULTS_DIR/'mixed_app'/'day5_aggregation_baseline_comparison_all_seeds.csv',index=False)

baseline_day5_summary=(baseline_day5_df.groupby(['Aggregation_Method','Model','Window_size']).agg(
    Accuracy_mean=('Accuracy','mean'),Accuracy_std=('Accuracy','std'),
    Precision_macro_mean=('Precision_macro','mean'),Precision_macro_std=('Precision_macro','std'),
    Recall_macro_mean=('Recall_macro','mean'),Recall_macro_std=('Recall_macro','std'),
    F1_macro_mean=('F1_macro','mean'),F1_macro_std=('F1_macro','std'),
    F1_weighted_mean=('F1_weighted','mean'),F1_weighted_std=('F1_weighted','std'),
    LogLoss_mean=('LogLoss','mean'),LogLoss_std=('LogLoss','std'),
    Windows_mean=('Windows','mean'),Evaluable_classes_mean=('Evaluable_classes','mean')).reset_index())
for metric in ['Accuracy','Precision_macro','Recall_macro','F1_macro','F1_weighted','LogLoss']:
    baseline_day5_summary[f'{metric}_mean_std']=baseline_day5_summary[f'{metric}_mean'].map(lambda x:f'{x:.4f}')+' ± '+baseline_day5_summary[f'{metric}_std'].fillna(0).map(lambda x:f'{x:.4f}')
baseline_day5_summary.to_csv(RESULTS_DIR/'mixed_app'/'day5_aggregation_baseline_summary_mean_std.csv',index=False)
display(baseline_day5_summary[['Aggregation_Method','Model','Window_size','Accuracy_mean_std','Precision_macro_mean_std','Recall_macro_mean_std','F1_macro_mean_std','F1_weighted_mean_std','LogLoss_mean_std','Windows_mean','Evaluable_classes_mean']])

# Direct locked-window difference: proposed probability averaging minus majority vote.
locked_compare=baseline_day5_summary[baseline_day5_summary.Window_size.eq(LOCKED_WINDOW)].pivot(index='Model',columns='Aggregation_Method',values='F1_macro_mean').reset_index()
if {'Probability_Average','Majority_Vote'}.issubset(locked_compare.columns):
    locked_compare['Macro_F1_difference_probability_minus_majority']=locked_compare['Probability_Average']-locked_compare['Majority_Vote']
locked_compare.to_csv(RESULTS_DIR/'mixed_app'/'day5_locked_probability_vs_majority_f1.csv',index=False)
display(locked_compare)


## Mixed-validation window-selection curves and primary time-aligned concurrent Day-5 comparison

The candidate-window curves use **time-aligned concurrent Day-4 validation**. Day 5 remains untouched until the common window is locked. The final comparison uses the time-aligned concurrent Day-5 stream for W=1 versus the locked W*.


In [ ]:
# ------------------------------------------------------------------
# VALIDATION CURVE — candidate-window analysis
# ------------------------------------------------------------------

for metric, ylabel, filename in [
    (
        'F1_macro',
        'Validation Macro F1',
        'mixed_validation_macro_f1_by_window_size.png'
    ),
    (
        'Accuracy',
        'Validation accuracy',
        'mixed_validation_accuracy_by_window_size.png'
    )
]:

    val_plot = (
        validation_metrics_df
        .groupby(
            [
                'Model',
                'Window_size'
            ]
        )[metric]
        .mean()
        .reset_index()
    )

    plt.figure(
        figsize=(9, 5)
    )

    for model_name, g in val_plot.groupby(
        'Model'
    ):

        g = g.sort_values(
            'Window_size'
        )

        plt.plot(
            g['Window_size'],
            g[metric],
            marker='o',
            label=model_name
        )

    plt.axvline(
        LOCKED_WINDOW,
        linestyle='--',
        linewidth=1,
        label=f'Locked W={LOCKED_WINDOW}'
    )

    plt.xlabel(
        'Multi-flow window size'
    )

    plt.ylabel(
        ylabel
    )

    plt.title(
        'Day-4 Mixed-Application Window Selection'
    )

    plt.legend(
        fontsize=8
    )

    plt.tight_layout()

    plt.savefig(
        RESULTS_DIR /
        'plots' /
        filename,
        dpi=200
    )

    plt.show()


# ------------------------------------------------------------------
# FINAL DAY-5 COMPARISON — only W=1 and locked W*
# ------------------------------------------------------------------

day5_plot = (
    metrics_df
    .groupby(
        [
            'Model',
            'Window_size'
        ]
    )[
        [
            'Accuracy',
            'F1_macro'
        ]
    ]
    .mean()
    .reset_index()
)

for metric, ylabel, filename in [
    (
        'F1_macro',
        'Day-5 Macro F1',
        'mixed_day5_single_vs_locked_macro_f1.png'
    ),
    (
        'Accuracy',
        'Day-5 accuracy',
        'mixed_day5_single_vs_locked_accuracy.png'
    )
]:

    pivot = day5_plot.pivot(
        index='Model',
        columns='Window_size',
        values=metric
    )

    pivot.plot(
        kind='bar',
        figsize=(10, 5)
    )

    plt.ylabel(
        ylabel
    )

    plt.xlabel(
        'Model'
    )

    plt.title(
        f'Primary Mixed Day-5 Test: '
        f'Single Flow vs Locked W={LOCKED_WINDOW}'
    )

    plt.xticks(
        rotation=30,
        ha='right'
    )

    plt.tight_layout()

    plt.savefig(
        RESULTS_DIR /
        'plots' /
        filename,
        dpi=200
    )

    plt.show()


## Normalized mean confusion matrices — primary time-aligned concurrent Day-5 conditions

Confusion matrices below are generated from the time-aligned concurrent Day-5 test for W=1 and the validation-selected locked window. Rows with no true support remain empty rather than being interpreted as classification failures.


In [ ]:
for (model_name,w), matrices in cms.items():
    mean_cm = np.mean(np.stack(matrices),axis=0)
    row_sums = mean_cm.sum(axis=1,keepdims=True)
    pct = np.divide(mean_cm,row_sums,out=np.zeros_like(mean_cm,dtype=float),where=row_sums!=0)*100

    plt.figure(figsize=(11,9))
    im = plt.imshow(pct,cmap='viridis',vmin=0,vmax=100)
    plt.colorbar(im,label='Percentage (%)')
    plt.xticks(np.arange(num_classes),class_names,rotation=45,ha='right')
    plt.yticks(np.arange(num_classes),class_names)
    plt.xlabel('Predicted class')
    plt.ylabel('True class')
    plt.title(f'{model_name} - Mixed Day-5 Mean Confusion Matrix (%) - {w} Flow(s)')
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j,i,f'{pct[i,j]:.1f}%',ha='center',va='center',fontsize=7,
                     color='white' if pct[i,j]>=50 else 'black')
    plt.tight_layout()
    safe = model_name.replace(' ','_')
    plt.savefig(RESULTS_DIR/'confusion_matrices'/f'mixed_{safe}_window{w}_percent.png',dpi=200)
    plt.show()

    pd.DataFrame(pct,index=class_names,columns=class_names).to_csv(
        RESULTS_DIR/'confusion_matrices'/f'mixed_{safe}_window{w}_percent.csv'
    )


## Primary time-aligned concurrent Day-5 per-class mean and standard deviation

Classes with no support at a given window are retained as N/A (`NaN`) with `Support=0`; they are not converted to zero-F1 observations.


In [ ]:
per_class_summary=per_class_df.groupby(['Model','Window_size','Class'],dropna=False).agg(
    Precision_mean=('Precision','mean'),Precision_std=('Precision','std'),Recall_mean=('Recall','mean'),Recall_std=('Recall','std'),
    F1_mean=('F1','mean'),F1_std=('F1','std'),Support_mean=('Support','mean'),Evaluable=('Evaluable','max')).reset_index()
per_class_summary.to_csv(RESULTS_DIR/'mixed_app'/'day5_mixed_primary_per_class_mean_std.csv',index=False); display(per_class_summary)


## Primary time-aligned concurrent Day-5 improvement over the single-flow baseline

This compares the validation-locked mixed-buffer operating point against W=1 on untouched mixed Day 5.


In [ ]:
mean_results = (
    metrics_df
    .groupby(
        [
            'Model',
            'Window_size'
        ]
    )[
        [
            'Accuracy',
            'F1_macro'
        ]
    ]
    .mean()
    .reset_index()
)

baseline = (
    mean_results[
        mean_results.Window_size.eq(1)
    ][
        [
            'Model',
            'Accuracy',
            'F1_macro'
        ]
    ]
    .rename(
        columns={
            'Accuracy':
                'Baseline_accuracy',
            'F1_macro':
                'Baseline_macro_F1'
        }
    )
)

locked_only = (
    mean_results[
        mean_results.Window_size.eq(
            LOCKED_WINDOW
        )
    ]
    .copy()
)

improvement = locked_only.merge(
    baseline,
    on='Model',
    how='left'
)

improvement[
    'Accuracy_gain'
] = (
    improvement['Accuracy']
    - improvement[
        'Baseline_accuracy'
    ]
)

improvement[
    'Macro_F1_gain'
] = (
    improvement['F1_macro']
    - improvement[
        'Baseline_macro_F1'
    ]
)

improvement[
    'Locked_window'
] = LOCKED_WINDOW

improvement.to_csv(
    RESULTS_DIR /
    'metrics' /
    'day5_mixed_locked_improvement_over_single_flow.csv',
    index=False
)

display(
    improvement.sort_values(
        'Model'
    )
)


## Locked evaluation protocol audit

This verifies complete-day separation, mixed Day-4-only selection, and that the primary time-aligned concurrent Day-5 test contains only W=1 and the validation-locked W*.


In [ ]:
assert set(metrics_df['Window_size'].unique()) == set(FINAL_TEST_WINDOWS)
assert set(FINAL_TEST_WINDOWS) == set([1,LOCKED_WINDOW])
assert int(common_window_scores.sort_values(['Common_validation_macro_F1','Window_size'],ascending=[False,True]).iloc[0]['Window_size']) == LOCKED_WINDOW
assert set(df.loc[df['_split'].eq('train'),'_day_number'].unique()) == {1,2,3}
assert set(df.loc[df['_split'].eq('validation'),'_day_number'].unique()) == {4}
assert set(df.loc[df['_split'].eq('test'),'_day_number'].unique()) == {5}
assert TEST_SELECTED_MODEL == validation_locked_model.iloc[0]['Model']
print(f'PASS: primary time-aligned concurrent Day 5 contains only W=1 and locked W={LOCKED_WINDOW}.')
print('PASS: complete-day train/validation/test separation is enforced.')
print('PASS: locked window and SHAP/LIME focal model were selected automatically from time-aligned concurrent Day-4 validation.')


## Post-hoc explainability of the automatically selected Day-4 validation model

The focal model is selected automatically as the highest-performing model at the locked window on **time-aligned concurrent Day-4 validation**. Day 5 never selects the model. SHAP and LIME remain flow-level explanations of correctly classified Day-5 flows from that pre-locked model, because the base classifier operates on individual flow features before mixed-buffer aggregation.


In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer

XAI_TOP_K = 15
SHAP_BACKGROUND = 200
SHAP_SAMPLES_PER_CLASS = 100
LIME_SAMPLES_PER_CLASS = 25
LIME_NUM_SAMPLES = 3000
XAI_SEED = 42

XAI_DIR = RESULTS_DIR / 'xai'
XAI_SHAP_DIR = XAI_DIR / 'shap'
XAI_LIME_DIR = XAI_DIR / 'lime'
XAI_AGREE_DIR = XAI_DIR / 'agreement'

for p in [XAI_DIR, XAI_SHAP_DIR, XAI_LIME_DIR, XAI_AGREE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('XAI output folder:', XAI_DIR)
print('Day-4 mixed-validation-selected focal model:', TEST_SELECTED_MODEL)


In [ ]:
# Refit the Day-4-locked focal model deterministically for post-hoc explanation.
# Model identity was fixed from Day-4 validation before Day-5 scoring.

selected_seed = XAI_SEED

prep_xai = make_preprocessor()

Xtr_xai = prep_xai.fit_transform(
    Xtr_raw
)

Xv_xai = prep_xai.transform(
    Xv_raw
)

Xte_xai = prep_xai.transform(
    Xte_raw
)

selected_model = build_models(
    selected_seed
)[TEST_SELECTED_MODEL]

selected_model = fit_model(
    TEST_SELECTED_MODEL,
    selected_model,
    Xtr_xai,
    ytr,
    Xv_xai,
    yv
)

feature_names = prep_xai.get_feature_names_out()

feature_names = np.asarray(
    [
        str(x)
        for x in feature_names
    ]
)

test_probs_xai = selected_model.predict_proba(
    Xte_xai
)

test_pred_xai = np.argmax(
    test_probs_xai,
    axis=1
)

print(
    'Day-4 mixed-validation-selected focal model fitted:',
    TEST_SELECTED_MODEL
)

print(
    'Locked common multi-flow window:',
    LOCKED_WINDOW
)

print(
    'Transformed feature count:',
    len(feature_names)
)

print(
    'Day-5 single-flow accuracy of focal model:',
    accuracy_score(
        yte,
        test_pred_xai
    )
)


In [ ]:
# Semantic feature mapping for fair SHAP-LIME comparison.
def semantic_feature_name(encoded_feature):
    name = str(encoded_feature)

    if name.startswith('num__'):
        return name[len('num__'):]

    if name.startswith('cat__'):
        remainder = name[len('cat__'):]
        for col in sorted(categorical_cols, key=len, reverse=True):
            if remainder == col or remainder.startswith(col + '_'):
                return col
        return remainder

    return name


def clean_feature_name(encoded_feature):
    name = str(encoded_feature)
    if name.startswith('num__'):
        return name[len('num__'):]
    if name.startswith('cat__'):
        return name[len('cat__'):]
    return name


In [ ]:
# SHAP: top 15 features for each class on correctly classified Day-5 flows.

rng = np.random.default_rng(XAI_SEED)

# Background drawn from training data only.
bg_n = min(SHAP_BACKGROUND, len(Xtr_xai))
bg_idx = rng.choice(len(Xtr_xai), size=bg_n, replace=False)
X_background = Xtr_xai[bg_idx]

try:
    explainer = shap.TreeExplainer(selected_model)
except Exception as exc:
    raise RuntimeError(
        f'SHAP TreeExplainer does not support the Day-4 validation-selected '
        f'model {TEST_SELECTED_MODEL}: {exc}. '
        'The benchmark results remain valid; only the XAI adapter would need '
        'model-specific handling.'
    )

shap_rows = []
shap_top_sets = {}
shap_semantic_sets = {}

for class_id, class_name in enumerate(class_names):
    candidates = np.where(
        (yte == class_id) &
        (test_pred_xai == class_id)
    )[0]

    if len(candidates) == 0:
        print(f'No correctly classified Day-5 flows for {class_name}; skipping SHAP.')
        shap_top_sets[class_name] = set()
        shap_semantic_sets[class_name] = set()
        continue

    take = min(SHAP_SAMPLES_PER_CLASS, len(candidates))
    chosen = rng.choice(candidates, size=take, replace=False)
    X_class = Xte_xai[chosen]

    shap_values = explainer.shap_values(X_class)

    # Handle multiclass TreeExplainer SHAP formats across SHAP versions.
    if isinstance(shap_values, list):
        class_shap = np.asarray(shap_values[class_id])
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3:
            if arr.shape[2] == num_classes:
                class_shap = arr[:, :, class_id]
            elif arr.shape[0] == num_classes:
                class_shap = arr[class_id]
            else:
                raise ValueError(f'Unexpected SHAP array shape: {arr.shape}')
        elif arr.ndim == 2:
            class_shap = arr
        else:
            raise ValueError(f'Unexpected SHAP array shape: {arr.shape}')

    mean_abs = np.mean(np.abs(class_shap), axis=0)
    order = np.argsort(mean_abs)[::-1][:XAI_TOP_K]

    exact_set = set()
    semantic_set = set()

    for rank, feat_idx in enumerate(order, start=1):
        feat = feature_names[int(feat_idx)]
        semantic = semantic_feature_name(feat)

        exact_set.add(feat)
        semantic_set.add(semantic)

        shap_rows.append({
            'Class': class_name,
            'Rank': rank,
            'Feature': clean_feature_name(feat),
            'Encoded_feature': feat,
            'Semantic_feature': semantic,
            'Mean_abs_SHAP': float(mean_abs[int(feat_idx)]),
            'Explained_correct_flows': int(take),
        })

    shap_top_sets[class_name] = exact_set
    shap_semantic_sets[class_name] = semantic_set

shap_table = pd.DataFrame(shap_rows)
shap_table.to_csv(
    XAI_SHAP_DIR / 'day5_validation_selected_model_shap_top15_per_class.csv',
    index=False
)

for cls in class_names:
    print('\n' + '=' * 100)
    print('SHAP -', TEST_SELECTED_MODEL, '-', cls)
    display(shap_table[shap_table['Class'] == cls])


In [ ]:
# LIME: top 15 features for each class on correctly classified Day-5 flows.
#
# Zero-inclusive aggregation:
# sum(abs(local weight)) / number of explained class instances.
# A feature absent from a local explanation contributes zero.

lime_explainer = LimeTabularExplainer(
    training_data=np.asarray(Xtr_xai),
    feature_names=feature_names.tolist(),
    class_names=class_names,
    mode='classification',
    discretize_continuous=True,
    random_state=XAI_SEED
)

lime_rows = []
lime_top_sets = {}
lime_semantic_sets = {}

for class_id, class_name in enumerate(class_names):
    candidates = np.where(
        (yte == class_id) &
        (test_pred_xai == class_id)
    )[0]

    if len(candidates) == 0:
        print(f'No correctly classified Day-5 flows for {class_name}; skipping LIME.')
        lime_top_sets[class_name] = set()
        lime_semantic_sets[class_name] = set()
        continue

    take = min(LIME_SAMPLES_PER_CLASS, len(candidates))
    chosen = rng.choice(candidates, size=take, replace=False)

    sums = {}
    occurrences = {}

    for idx in chosen:
        exp = lime_explainer.explain_instance(
            np.asarray(Xte_xai[idx]),
            selected_model.predict_proba,
            labels=[class_id],
            num_features=XAI_TOP_K,
            num_samples=LIME_NUM_SAMPLES
        )

        for feat_idx, weight in exp.local_exp[class_id]:
            feat = feature_names[int(feat_idx)]
            sums[feat] = sums.get(feat, 0.0) + abs(float(weight))
            occurrences[feat] = occurrences.get(feat, 0) + 1

    averaged = {
        feat: total / take
        for feat, total in sums.items()
    }

    ranked = sorted(
        averaged.items(),
        key=lambda x: x[1],
        reverse=True
    )[:XAI_TOP_K]

    exact_set = set()
    semantic_set = set()

    for rank, (feat, importance) in enumerate(ranked, start=1):
        semantic = semantic_feature_name(feat)
        exact_set.add(feat)
        semantic_set.add(semantic)

        lime_rows.append({
            'Class': class_name,
            'Rank': rank,
            'Feature': clean_feature_name(feat),
            'Encoded_feature': feat,
            'Semantic_feature': semantic,
            'Mean_abs_LIME_zero_inclusive': float(importance),
            'Occurrence_count': int(occurrences.get(feat, 0)),
            'Explained_correct_flows': int(take),
            'Occurrence_rate': float(occurrences.get(feat, 0) / take),
        })

    lime_top_sets[class_name] = exact_set
    lime_semantic_sets[class_name] = semantic_set

lime_table = pd.DataFrame(lime_rows)
lime_table.to_csv(
    XAI_LIME_DIR / 'day5_validation_selected_model_lime_top15_per_class.csv',
    index=False
)

for cls in class_names:
    print('\n' + '=' * 100)
    print('LIME -', TEST_SELECTED_MODEL, '-', cls)
    display(lime_table[lime_table['Class'] == cls])


In [ ]:
# SHAP-LIME agreement: exact encoded features and semantic/original feature families.

agreement_rows = []

for cls in class_names:
    shap_exact = shap_top_sets.get(cls, set())
    lime_exact = lime_top_sets.get(cls, set())
    shap_sem = shap_semantic_sets.get(cls, set())
    lime_sem = lime_semantic_sets.get(cls, set())

    exact_union = shap_exact | lime_exact
    semantic_union = shap_sem | lime_sem

    exact_jaccard = (
        len(shap_exact & lime_exact) / len(exact_union)
        if exact_union else np.nan
    )
    semantic_jaccard = (
        len(shap_sem & lime_sem) / len(semantic_union)
        if semantic_union else np.nan
    )

    agreement_rows.append({
        'Class': cls,
        'Exact_Jaccard': exact_jaccard,
        'Exact_common_count': len(shap_exact & lime_exact),
        'Semantic_Jaccard': semantic_jaccard,
        'Semantic_common_count': len(shap_sem & lime_sem),
        'Exact_common_features': '; '.join(sorted(shap_exact & lime_exact)),
        'Semantic_common_features': '; '.join(sorted(shap_sem & lime_sem)),
    })

agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv(
    XAI_AGREE_DIR / 'day5_validation_selected_model_shap_lime_agreement_by_class.csv',
    index=False
)

display(agreement_df)

print('\nMean exact Jaccard:',
      agreement_df['Exact_Jaccard'].mean())
print('Mean semantic Jaccard:',
      agreement_df['Semantic_Jaccard'].mean())


In [ ]:
# Agreement graph.
x = np.arange(len(class_names))
width = 0.36

plt.figure(figsize=(12, 6))
plt.bar(
    x - width / 2,
    agreement_df['Exact_Jaccard'],
    width,
    label='Exact encoded features'
)
plt.bar(
    x + width / 2,
    agreement_df['Semantic_Jaccard'],
    width,
    label='Semantic feature families'
)

plt.xlabel('Application class')
plt.ylabel('Jaccard agreement')
plt.title('Day-4 Validation-Selected Model SHAP-LIME Top-15 Agreement')
plt.xticks(x, class_names, rotation=45, ha='right')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()

plot_path = XAI_AGREE_DIR / 'day5_validation_selected_model_shap_lime_top15_agreement.png'
plt.savefig(plot_path, dpi=200)
plt.show()

print('Agreement graph saved to:', plot_path)
